# CNN & LSTM all tasks — robustness evaluation

This notebook mirrors `mlp_all_tasks_robustness.ipynb`, but runs CNN and LSTM models for all three tasks.

Workflow:

1. Run `cnn_lstm_hyperparameter_tuning.ipynb`.
2. Copy the printed `FINAL_TUNED_CFGS` dictionary.
3. Paste it into the config cell below.
4. Run this robustness notebook.

This removes stale hard-coded training settings and keeps the robustness evaluation aligned with the final fine-tuning run.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root:", project_root)


Project root: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [3]:
import numpy as np
import pandas as pd
import torch

from src.data_prep import prepare_uji_data, prepare_uji_data_downgradation as prepare_uji_data_for_robustness
from src.degradation_experiments import (
    COMPOSITE_SEED_BASE,
    COMPOSITE_TRAIN_SCENARIOS,
    EVAL_DEGRADATION_SCENARIOS as EVAL_DEGRADATION_SCENARIOS,
    eval_scenario_seed,
)
from src.models import (
    CNNCoordinateModel, CNNJointModel, CNNMultiTaskModel,
    LSTMCoordinateModel, LSTMJointModel, LSTMMultiTaskModel,
)
from src.training import TrainConfig, evaluate_on_tensors, train_from_tensors


In [4]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = bundle.X_train.shape[1]

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

# Composite training data: clean + controlled perturbed copies
X_train_parts = [bundle.X_train]
for k, spec in enumerate(COMPOSITE_TRAIN_SCENARIOS):
    deg = prepare_uji_data_for_robustness(bundle, seed=COMPOSITE_SEED_BASE + k, **spec)
    X_train_parts.append(deg.X_train)

X_train_composite = np.concatenate(X_train_parts, axis=0)
n_parts = len(X_train_parts)
joint_y_composite = np.concatenate([joint_y_train] * n_parts, axis=0)
mt_y_composite = np.concatenate([mt_y_train] * n_parts, axis=0)
coord_y_composite = np.concatenate([coord_y_train] * n_parts, axis=0)

print("device:", device)
print("robustness scenarios:", len(EVAL_DEGRADATION_SCENARIOS))
print("X_train clean / composite:", bundle.X_train.shape[0], X_train_composite.shape[0])


device: cuda
robustness scenarios: 18
X_train clean / composite: 19937 79748


## Paste final fine-tuned settings

Copy `FINAL_TUNED_CFGS` from `fair_cnn_lstm_hyperparameter_tuning.ipynb` and paste it below.

Expected shape:

```python
FINAL_TUNED_CFGS = {
    "cnn": {
        "joint": {"lr": ..., "weight_decay": ..., "max_epochs": ..., "patience": ..., "batch_size": ..., "val_batch_size": ..., ...},
        "multitask": {...},
        "coordinate": {...},
    },
    "lstm": {
        "joint": {...},
        "multitask": {...},
        "coordinate": {...},
    },
}
```


In [5]:
MANUAL_CONFIG_FAMILIES = ("cnn", "lstm")
TASKS = ("joint", "multitask", "coordinate")
TRAIN_KEYS = (
    "lr",
    "weight_decay",
    "max_epochs",
    "patience",
    "print_every",
    "batch_size",
    "val_batch_size",
    "grad_clip_norm",
)

# Paste the FINAL_TUNED_CFGS dictionary printed by the matching fine-tuning notebook here.
#
# Expected shape:
# FINAL_TUNED_CFGS = {
#     "mlp": {  # or "cnn" / "lstm"
#         "joint": {"lr": ..., "weight_decay": ..., "grad_clip_norm": ..., "max_epochs": ..., "patience": ..., "print_every": 5, "batch_size": 256, "val_batch_size": 512},
#         "multitask": {...},
#         "coordinate": {...},
#     }
# }

FINAL_TUNED_CFGS ={
    'cnn': {'coordinate': {'lr': 0.0005,
   'weight_decay': 0.0001,
   'grad_clip_norm': None,
   'max_epochs': 60,
   'patience': 12,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0001,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}},
 'lstm': {'coordinate': {'lr': 0.001,
   'weight_decay': 0.0001,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.0005,
   'weight_decay': 0.0001,
   'grad_clip_norm': 1.0,
   'max_epochs': 60,
   'patience': 12,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}}
 }

def validate_manual_training_configs(
    final_cfgs: dict,
    families: tuple[str, ...],
    tasks: tuple[str, ...] = TASKS,
) -> pd.DataFrame:
    if not final_cfgs:
        raise RuntimeError(
            "FINAL_TUNED_CFGS is empty. Copy the final dictionary from "
            "fair_cnn_lstm_hyperparameter_tuning_v3_original_names.ipynb first."
        )

    missing = []
    rows = []
    for family in families:
        if family not in final_cfgs:
            missing.append((family, "<family missing>"))
            continue
        for task in tasks:
            if task not in final_cfgs[family]:
                missing.append((family, task))
                continue
            spec = dict(final_cfgs[family][task])
            required = ("lr", "weight_decay", "max_epochs", "patience")
            missing_keys = [k for k in required if k not in spec]
            if missing_keys:
                raise RuntimeError(f"Config for {(family, task)} is missing keys: {missing_keys}")

            rows.append({"family": family, "task": task, **{k: spec.get(k) for k in TRAIN_KEYS}})

    if missing:
        raise RuntimeError(f"Missing required tuned configs: {missing}")

    return pd.DataFrame(rows).sort_values(["family", "task"]).reset_index(drop=True)

selected_cfg_df = validate_manual_training_configs(FINAL_TUNED_CFGS, MANUAL_CONFIG_FAMILIES)
selected_cfg_df


,family,task,lr,weight_decay,max_epochs,patience,print_every,batch_size,val_batch_size,grad_clip_norm
0,cnn,coordinate,0.0005,0.0001,60,12,5,256,512,NaN
1,cnn,joint,0.0010,0.0005,50,10,5,256,512,NaN
2,cnn,multitask,0.0010,0.0001,50,10,5,256,512,NaN
3,lstm,coordinate,0.0010,0.0001,50,10,5,256,512,1.0
4,lstm,joint,0.0005,0.0005,80,15,5,256,512,1.0
5,lstm,multitask,0.0005,0.0001,60,12,5,256,512,1.0


In [6]:
def make_cfg(spec: dict, run_name: str) -> TrainConfig:
    return TrainConfig(
        run_name=run_name,
        lr=spec["lr"],
        weight_decay=spec["weight_decay"],
        batch_size=spec.get("batch_size", 256),
        val_batch_size=spec.get("val_batch_size", 512),
        max_epochs=spec["max_epochs"],
        patience=spec["patience"],
        print_every=spec.get("print_every", 5),
        grad_clip_norm=spec.get("grad_clip_norm", None),
    )

def eval_robustness_grid(
    model: torch.nn.Module,
    y_val: np.ndarray,
    eval_batch_size: int,
) -> pd.DataFrame:
    rows = []
    for i, (name, dr, bd, ns) in enumerate(EVAL_DEGRADATION_SCENARIOS):
        seed = eval_scenario_seed(i)
        if name == "clean":
            b = bundle
        else:
            b = prepare_uji_data_for_robustness(
                bundle,
                dropout_rate=dr,
                bias_db=bd,
                noise_std=ns,
                seed=seed,
            )
        m = evaluate_on_tensors(
            model, b.X_val, y_val, device, batch_size=eval_batch_size
        )
        rows.append({"scenario": name, **dict(m)})
    return pd.DataFrame(rows)

def stack_model_results(models, y_vals, eval_batch_sizes, train_regime):
    parts = []
    for mname, model in models.items():
        df = eval_robustness_grid(model, y_vals[mname], eval_batch_sizes[mname])
        df["model"] = mname
        df["train_regime"] = train_regime
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


## 1) Train on clean data


In [7]:
# CNN clean
cnn_joint_clean = CNNJointModel(in_dim=in_dim)
train_from_tensors(
    cnn_joint_clean,
    bundle.X_train, joint_y_train,
    bundle.X_val, joint_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["cnn"]["joint"], "cnn_joint_clean"),
)

cnn_multitask_clean = CNNMultiTaskModel(in_dim=in_dim)
train_from_tensors(
    cnn_multitask_clean,
    bundle.X_train, mt_y_train,
    bundle.X_val, mt_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["cnn"]["multitask"], "cnn_multitask_clean"),
)

cnn_coordinate_clean = CNNCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)
train_from_tensors(
    cnn_coordinate_clean,
    bundle.X_train, coord_y_train,
    bundle.X_val, coord_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["cnn"]["coordinate"], "cnn_coordinate_clean"),
)

# LSTM clean
lstm_joint_clean = LSTMJointModel(in_dim=in_dim)
train_from_tensors(
    lstm_joint_clean,
    bundle.X_train, joint_y_train,
    bundle.X_val, joint_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["lstm"]["joint"], "lstm_joint_clean"),
)

lstm_multitask_clean = LSTMMultiTaskModel(in_dim=in_dim)
train_from_tensors(
    lstm_multitask_clean,
    bundle.X_train, mt_y_train,
    bundle.X_val, mt_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["lstm"]["multitask"], "lstm_multitask_clean"),
)

lstm_coordinate_clean = LSTMCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)
train_from_tensors(
    lstm_coordinate_clean,
    bundle.X_train, coord_y_train,
    bundle.X_val, coord_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["lstm"]["coordinate"], "lstm_coordinate_clean"),
)

models_clean = {
    "cnn_joint": cnn_joint_clean,
    "cnn_multitask": cnn_multitask_clean,
    "cnn_coordinate": cnn_coordinate_clean,
    "lstm_joint": lstm_joint_clean,
    "lstm_multitask": lstm_multitask_clean,
    "lstm_coordinate": lstm_coordinate_clean,
}

yval_by_model = {
    "cnn_joint": joint_y_val,
    "cnn_multitask": mt_y_val,
    "cnn_coordinate": coord_y_val,
    "lstm_joint": joint_y_val,
    "lstm_multitask": mt_y_val,
    "lstm_coordinate": coord_y_val,
}

eval_batch_sizes = {
    "cnn_joint": FINAL_TUNED_CFGS["cnn"]["joint"]["val_batch_size"],
    "cnn_multitask": FINAL_TUNED_CFGS["cnn"]["multitask"]["val_batch_size"],
    "cnn_coordinate": FINAL_TUNED_CFGS["cnn"]["coordinate"]["val_batch_size"],
    "lstm_joint": FINAL_TUNED_CFGS["lstm"]["joint"]["val_batch_size"],
    "lstm_multitask": FINAL_TUNED_CFGS["lstm"]["multitask"]["val_batch_size"],
    "lstm_coordinate": FINAL_TUNED_CFGS["lstm"]["coordinate"]["val_batch_size"],
}

results_clean = stack_model_results(models_clean, yval_by_model, eval_batch_sizes, "clean_train")
results_clean


epoch=001 train_loss=1.3518 val_loss=1.6485 score=0.4140
epoch=005 train_loss=0.2166 val_loss=1.0689 score=0.6742
epoch=010 train_loss=0.0974 val_loss=0.9817 score=0.7462
epoch=015 train_loss=0.0615 val_loss=0.9423 score=0.7678
epoch=020 train_loss=0.0294 val_loss=0.8576 score=0.8065
epoch=025 train_loss=0.0255 val_loss=0.9084 score=0.7984
epoch=030 train_loss=0.0230 val_loss=0.8845 score=0.8173
epoch=035 train_loss=0.0132 val_loss=0.8853 score=0.8281
epoch=040 train_loss=0.0150 val_loss=0.9982 score=0.8047
epoch=045 train_loss=0.0100 val_loss=0.9293 score=0.8299
epoch=050 train_loss=0.0096 val_loss=0.9226 score=0.8335
epoch=001 train_loss=1.5062 val_loss=1.8408 score=0.3375
epoch=005 train_loss=0.3143 val_loss=1.7026 score=0.5734
epoch=010 train_loss=0.1505 val_loss=1.2046 score=0.6742
epoch=015 train_loss=0.0929 val_loss=1.0550 score=0.7633
epoch=020 train_loss=0.0674 val_loss=1.0947 score=0.7624
epoch=025 train_loss=0.0341 val_loss=0.9868 score=0.8164
epoch=030 train_loss=0.0328 val

,scenario,score,joint_accuracy,building_accuracy,floor_accuracy,eval_loss,model,train_regime,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,clean,0.833483,0.833483,0.954095,0.843384,0.922597,cnn_joint,clean_train,NaN,NaN,NaN,NaN
1,dropout_0.05,0.769577,0.769577,0.919892,0.785779,1.342968,cnn_joint,clean_train,NaN,NaN,NaN,NaN
2,dropout_0.10,0.709271,0.709271,0.878488,0.740774,1.839081,cnn_joint,clean_train,NaN,NaN,NaN,NaN
3,dropout_0.15,0.621062,0.621062,0.823582,0.673267,2.474757,cnn_joint,clean_train,NaN,NaN,NaN,NaN
4,dropout_0.20,0.584158,0.584158,0.783078,0.647165,3.137532,cnn_joint,clean_train,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
103,drop0.35_bias4_noise2,-18.884920,NaN,NaN,NaN,0.053716,lstm_coordinate,clean_train,0.207311,0.327767,18.884920,29.826238
104,drop0.40_bias5_noise3,-20.619098,NaN,NaN,NaN,0.052846,lstm_coordinate,clean_train,0.224031,0.325104,20.619098,29.819274
105,drop0.15_bias6_noise0,-13.903763,NaN,NaN,NaN,0.025005,lstm_coordinate,clean_train,0.155142,0.223628,13.903763,20.023100
106,bias5_only,-11.883252,NaN,NaN,NaN,0.017272,lstm_coordinate,clean_train,0.131662,0.185863,11.883252,16.814124


## 2) Train on augmented/composite data


In [8]:
# CNN augmented
cnn_joint_aug = CNNJointModel(in_dim=in_dim)
train_from_tensors(
    cnn_joint_aug,
    X_train_composite, joint_y_composite,
    bundle.X_val, joint_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["cnn"]["joint"], "cnn_joint_aug"),
)

cnn_multitask_aug = CNNMultiTaskModel(in_dim=in_dim)
train_from_tensors(
    cnn_multitask_aug,
    X_train_composite, mt_y_composite,
    bundle.X_val, mt_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["cnn"]["multitask"], "cnn_multitask_aug"),
)

cnn_coordinate_aug = CNNCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)
train_from_tensors(
    cnn_coordinate_aug,
    X_train_composite, coord_y_composite,
    bundle.X_val, coord_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["cnn"]["coordinate"], "cnn_coordinate_aug"),
)

# LSTM augmented
lstm_joint_aug = LSTMJointModel(in_dim=in_dim)
train_from_tensors(
    lstm_joint_aug,
    X_train_composite, joint_y_composite,
    bundle.X_val, joint_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["lstm"]["joint"], "lstm_joint_aug"),
)

lstm_multitask_aug = LSTMMultiTaskModel(in_dim=in_dim)
train_from_tensors(
    lstm_multitask_aug,
    X_train_composite, mt_y_composite,
    bundle.X_val, mt_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["lstm"]["multitask"], "lstm_multitask_aug"),
)

lstm_coordinate_aug = LSTMCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)
train_from_tensors(
    lstm_coordinate_aug,
    X_train_composite, coord_y_composite,
    bundle.X_val, coord_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["lstm"]["coordinate"], "lstm_coordinate_aug"),
)

models_aug = {
    "cnn_joint": cnn_joint_aug,
    "cnn_multitask": cnn_multitask_aug,
    "cnn_coordinate": cnn_coordinate_aug,
    "lstm_joint": lstm_joint_aug,
    "lstm_multitask": lstm_multitask_aug,
    "lstm_coordinate": lstm_coordinate_aug,
}

results_aug = stack_model_results(models_aug, yval_by_model, eval_batch_sizes, "aug_train")
results_aug


epoch=001 train_loss=1.3314 val_loss=1.2861 score=0.5608
epoch=005 train_loss=0.4247 val_loss=0.7936 score=0.7507
epoch=010 train_loss=0.3228 val_loss=0.6887 score=0.7930
epoch=015 train_loss=0.2395 val_loss=0.6285 score=0.8227
epoch=020 train_loss=0.2152 val_loss=0.7549 score=0.8308
epoch=025 train_loss=0.1704 val_loss=0.6304 score=0.8434
epoch=001 train_loss=1.5328 val_loss=1.5151 score=0.4320
epoch=005 train_loss=0.5334 val_loss=0.7817 score=0.7750
epoch=010 train_loss=0.4029 val_loss=0.8080 score=0.7651
epoch=015 train_loss=0.3471 val_loss=0.8240 score=0.7759
epoch=020 train_loss=0.3165 val_loss=0.8286 score=0.8038
epoch=025 train_loss=0.2882 val_loss=0.8186 score=0.7759
epoch=030 train_loss=0.2671 val_loss=0.8306 score=0.8038
epoch=035 train_loss=0.2136 val_loss=0.7176 score=0.8281
epoch=040 train_loss=0.1824 val_loss=0.7662 score=0.8416
epoch=045 train_loss=0.1703 val_loss=0.7978 score=0.8281
epoch=050 train_loss=0.1528 val_loss=0.8208 score=0.8389
epoch=001 train_loss=0.5907 val

,scenario,score,joint_accuracy,building_accuracy,floor_accuracy,eval_loss,model,train_regime,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,clean,0.853285,0.853285,0.965797,0.864987,0.592352,cnn_joint,aug_train,NaN,NaN,NaN,NaN
1,dropout_0.05,0.831683,0.831683,0.955896,0.846085,0.636857,cnn_joint,aug_train,NaN,NaN,NaN,NaN
2,dropout_0.10,0.811881,0.811881,0.936094,0.826283,0.698560,cnn_joint,aug_train,NaN,NaN,NaN,NaN
3,dropout_0.15,0.756076,0.756076,0.907291,0.776778,0.878050,cnn_joint,aug_train,NaN,NaN,NaN,NaN
4,dropout_0.20,0.738974,0.738974,0.891989,0.764176,0.950510,cnn_joint,aug_train,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
103,drop0.35_bias4_noise2,-12.771818,NaN,NaN,NaN,0.027115,lstm_coordinate,aug_train,0.143514,0.232873,12.771818,20.566487
104,drop0.40_bias5_noise3,-13.399100,NaN,NaN,NaN,0.021757,lstm_coordinate,aug_train,0.150723,0.208601,13.399100,18.277398
105,drop0.15_bias6_noise0,-11.401492,NaN,NaN,NaN,0.016184,lstm_coordinate,aug_train,0.128814,0.179909,11.401492,15.379797
106,bias5_only,-10.973864,NaN,NaN,NaN,0.015570,lstm_coordinate,aug_train,0.124618,0.176463,10.973864,14.928398


## 3) Compare `score` clean-train vs augmented-train


In [9]:
cmp = results_clean[["scenario", "model", "score"]].merge(
    results_aug[["scenario", "model", "score"]],
    on=["scenario", "model"],
    suffixes=("_clean_train", "_aug_train"),
)
cmp["delta_score"] = cmp["score_aug_train"] - cmp["score_clean_train"]
cmp


,scenario,model,score_clean_train,score_aug_train,delta_score
0,clean,cnn_joint,0.833483,0.853285,0.019802
1,dropout_0.05,cnn_joint,0.769577,0.831683,0.062106
2,dropout_0.10,cnn_joint,0.709271,0.811881,0.102610
3,dropout_0.15,cnn_joint,0.621062,0.756076,0.135014
4,dropout_0.20,cnn_joint,0.584158,0.738974,0.154815
...,...,...,...,...,...
103,drop0.35_bias4_noise2,lstm_coordinate,-18.884920,-12.771818,6.113103
104,drop0.40_bias5_noise3,lstm_coordinate,-20.619098,-13.399100,7.219998
105,drop0.15_bias6_noise0,lstm_coordinate,-13.903763,-11.401492,2.502271
106,bias5_only,lstm_coordinate,-11.883252,-10.973864,0.909388


In [10]:
cmp_pivot = cmp.pivot_table(
    index="scenario",
    columns="model",
    values="delta_score",
    aggfunc="first",
)
cmp_pivot


model,cnn_coordinate,cnn_joint,cnn_multitask,lstm_coordinate,lstm_joint,lstm_multitask
scenario,,,,,,
bias5_only,-1.466713,0.018002,0.049505,0.909388,0.010801,-0.002700
clean,-1.496448,0.019802,0.017102,1.220190,0.008101,-0.001800
drop0.15_bias6_noise0,16.213175,0.143114,0.170117,2.502271,0.033303,0.030603
drop0.25_bias3_noise1,27.539401,0.179118,0.234923,4.884804,0.053105,0.069307
drop0.35_bias4_noise2,37.329685,0.247525,0.294329,6.113103,0.080108,0.086409
drop0.40_bias5_noise3,38.870793,0.249325,0.277228,7.219998,0.105311,0.129613
dropout_0.05,4.359051,0.062106,0.077408,1.988675,0.024302,0.013501
dropout_0.10,10.779135,0.102610,0.121512,2.967819,0.040504,0.019802
dropout_0.15,19.784062,0.135014,0.168317,3.692995,0.054905,0.047705


In [11]:
# Optional: save the same artifacts used by result notebooks/plot scripts.
OUT_DIR = Path("notebooks/logs/cnn_lstm_robustness") if (Path.cwd().name != "notebooks") else Path("logs/cnn_lstm_robustness")
OUT_DIR.mkdir(parents=True, exist_ok=True)

selected_cfg_df.to_csv(OUT_DIR / "tuned_hparams.csv", index=False)
results_clean.to_csv(OUT_DIR / "results_clean.csv", index=False)
results_aug.to_csv(OUT_DIR / "results_aug.csv", index=False)
cmp.to_csv(OUT_DIR / "cmp_delta_score.csv", index=False)
cmp_pivot.to_csv(OUT_DIR / "cmp_delta_score_pivot.csv")

print("Saved CSV outputs to:", OUT_DIR.resolve())


Saved CSV outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/cnn_lstm_robustness
